# Notebook 34 - Second corpus, part 3: the recovery factorial on IoMT

The same five-method by five-seed by two-regime factorial as CICIoT2023, on the frozen NB33 structures, with everything notebook 31 taught built in from the start: a dual-mode BatchNorm audit at every epoch, every final checkpoint saved, one fixed recalibration applied to every final model, and every analysis reported under both normalisations with the recalibrated one as primary. The dense unpruned model is recovered under the identical conditions as a control, and every collapse snapshot is treated with the remedy and evaluated normally.

**Collapse definition, fixed before any result.** Eval-mode benign escalation exceeds batch-statistics benign escalation by more than 0.10 absolute. The CICIoT2023 absolute rule (eval above 0.10 with batch at or below 0.10) would fire on a healthy IoMT shallow student because that teacher already escalates 8.7%.

**Pre-registered claims** (stage 2): B1 equalisation index at least 0.70 in every cell; B2a a recovery-seed effect detectable in at least two of four cells; B2b the largest paired method difference below 0.03 AWBIR in every cell; B3 mean pairwise rank correlation across seeds below 0.5 in every cell; B4 at least one collapse; B5 every collapse remedied by recalibration. Held or not held, reported either way; a claim that does not hold changes the paper's generality statement.

**Staging.** `RUN_CELLS` in stage 2 orders the cells lightest first; the shallow full-recovery cells are the heavy half. Stages 4 and 5 resume per completed cell. Checkpoints go to `checkpoints/` and collapse snapshots to `collapse_snapshots/` under the results folder; both are gitignored, and the SHA-256 registry is what the repo keeps.

**Stages.** 1 bootstrap, 2 pre-registration, 3 bridge, teachers, graph, helpers with runtime proofs, 4 factorial, 5 unpruned control, 6 remedy, 7 analysis under both normalisations, 8 verdict, 9 figures. GPU required.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, hashlib, copy, itertools
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
from src.saber.bridge_iomt import build_bridge
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.surgery import prune_cnn1d_channels

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
IN32, IN33 = R / "32_iomt_bridge", R / "33_iomt_scores_structures"
OUT = R / "34_iomt_factorial"; OUT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUT / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)      # gitignored; SHA-256 in the registry
SNAP_DIR = OUT / "collapse_snapshots"; SNAP_DIR.mkdir(exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"; MODEL_DIR = REPO / "models/iomt"
print("device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration and run configuration
PREREG = {
    "arm": "B_second_corpus_factorial",
    "design": ("5 methods x 5 seeds x 2 architectures x 2 regimes on the frozen NB33 40% structures. Shallow: "
               "minimal_ce (1 unit on a seeded 10% subset) and full_ce (6 fixed full epochs). Deep: minimal_ce and "
               "standard_ce (2 full epochs). Adam 1e-3, batch 1024, class weighting with each teacher's own exponent. "
               "Dual-mode audit at EVERY epoch; every final checkpoint saved; ONE fixed BatchNorm recalibration "
               "(50 batches from a single training slice, seed 2026) applied to every final model; all analyses "
               "reported under both normalisations, recalibrated as primary"),
    "collapse_definition": ("normalisation collapse: eval-mode benign escalation exceeds batch-statistics benign "
                            "escalation by more than 0.10 absolute. Chosen before any result because the shallow IoMT "
                            "teacher's own baseline is 0.087, so the CICIoT2023 absolute 0.10 rule would fire on a "
                            "healthy student"),
    "controls": ("unpruned control: each dense teacher recovered under the identical seeded subsets, orders, weighting "
                 "and schedule for 8 (shallow) / 6 (deep) units with the dual-mode audit; remedy: every collapse "
                 "snapshot recalibrated on the fixed slice and evaluated normally"),
    "claims": {
        "B1": "equalisation: recovery equalisation index for recalibrated AWBIR >= 0.70 in all four cells",
        "B2a": "a recovery-seed effect on recalibrated AWBIR is detectable by permutation (p < 0.05) in >= 2 of 4 cells",
        "B2b": ("selection effects are small: the largest paired mean difference between methods in recalibrated AWBIR "
                "is below 0.03 (the materiality line of Section 4) in every cell"),
        "B3": "rank instability: mean pairwise rank correlation across seeds on recalibrated AWBIR < 0.5 in every cell",
        "B4": "at least one normalisation collapse occurs across the factorial epochs and the unpruned control",
        "B5": ("if B4 holds, recalibration brings every collapse snapshot's eval-mode benign escalation to within 0.02 "
               "of its batch-statistics value")},
    "reporting_rule": "each claim reported as held or not held; a claim that does not hold changes the generality statement",
    "seeds": [101, 211, 307, 401, 503], "no_test_access": True, "no_new_selection": True,
}
(OUT / "B34_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))

METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
SEEDS = PREREG["seeds"]
REGIMES = {"shallow": {"minimal_ce": ("subset", 1), "full_ce": ("full", 6)},
           "deep":    {"minimal_ce": ("subset", 1), "standard_ce": ("full", 2)}}
RUN_CELLS = [("shallow", "minimal_ce"), ("deep", "minimal_ce"), ("deep", "standard_ce"), ("shallow", "full_ce")]
SUBSET_FRACTION = 0.10; COLLAPSE_GAP = 0.10; CAL_SEED = 2026; MIN_W = 8
E_MAX = {"shallow": 8, "deep": 6}


In [ ]:
# Stage 3 - bridge, teachers, graph, evaluation subsample, helpers
TRAIN_LOADER, VAL_LOADER, CLASS_NAMES, taxonomy, MANIFEST = build_bridge(DATA_CACHE)
N_CLASSES = len(CLASS_NAMES)
robust_graph = pd.read_csv(IN32 / "asvg_edges_robust.csv")
FAM = np.asarray(taxonomy.class_to_family_index); FAMILIES = list(taxonomy.families)


class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


ARCHS = {"shallow": CNN1D, "deep": DeepCNN1D}
TEACHERS, TEACHER_ALPHA = {}, {}
for arch, cls in ARCHS.items():
    payload = torch.load(MODEL_DIR / f"{arch}_teacher_seed0.pt", map_location="cpu", weights_only=False)
    m = cls(N_CLASSES); m.load_state_dict(payload["state_dict"]); TEACHERS[arch] = m.to(DEVICE).eval()
    TEACHER_ALPHA[arch] = float(payload["alpha"])

Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
N_TRAIN = len(TRAIN_LOADER.dataset)
_train_y = TRAIN_LOADER.dataset.tensors[1].numpy(); COUNTS = np.bincount(_train_y, minlength=N_CLASSES)


def class_weights(alpha):
    w = np.zeros(N_CLASSES, dtype=np.float64); nz = COUNTS > 0
    w[nz] = 1.0 / np.power(COUNTS[nz], alpha); w[nz] /= w[nz].mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


CLASS_W = {a: class_weights(TEACHER_ALPHA[a]) for a in ARCHS}
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)


def forward_logits(model):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()


def forward_logits_batch_stats(model):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone())
             for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved:
            m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]
            m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}
T_PRED = {a: T_LOGITS[a].argmax(1) for a in TEACHERS}


def audit_from_logits(lg, arch):
    a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph, weight_column="robust_weight")
    p = lg.argmax(1); s_ok = FAM[p] == FAM[EX_Y]; t_ok = FAM[T_PRED[arch]] == FAM[EX_Y]
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]),
            "family_f1": float(a["family_macro_f1"]), "fine_f1": float(a["fine_macro_f1"]), "awbir": float(aw),
            "ece": float(a["ece15"]), "hsr_miss": float(a["hsr_miss_sensitive"]), "hsr_balanced": float(a["hsr_balanced_soc"]),
            "hsr_fatigue": float(a["hsr_alert_fatigue"]),
            "teacher_correction_family": float(np.mean(s_ok & ~t_ok)), "teacher_degradation_family": float(np.mean(~s_ok & t_ok))}


def audit(model, arch, mode="eval"):
    return audit_from_logits(forward_logits(model) if mode == "eval" else forward_logits_batch_stats(model), arch)


def set_bn_momentum(model, momentum):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.momentum = momentum


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches:
                break
            model(xb.to(DEVICE))
    set_bn_momentum(model, 0.1); model.eval()
    return model


def raw_student(arch, method):
    rm = pd.read_csv(IN33 / f"{arch}_{method}_r40_removed_groups.csv")
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist()) for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W)
    return st.to(DEVICE)


def make_subset_loader(seed):
    gen = torch.Generator().manual_seed(seed)
    sub = torch.randperm(N_TRAIN, generator=gen)[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=1024,
                                       shuffle=True, generator=torch.Generator().manual_seed(seed))


def is_collapse(ev, bs):
    return bool(ev["b2a"] - bs["b2a"] > COLLAPSE_GAP)


# runtime proofs
for _a, _m in TEACHERS.items():
    _b = {n: (x.running_mean.clone(), x.running_var.clone()) for n, x in _m.named_modules() if isinstance(x, nn.BatchNorm1d)}
    _e1 = forward_logits(_m); _p = forward_logits_batch_stats(_m); _e2 = forward_logits(_m)
    for n, x in _m.named_modules():
        if isinstance(x, nn.BatchNorm1d):
            assert torch.equal(x.running_mean, _b[n][0]) and torch.equal(x.running_var, _b[n][1])
    assert np.allclose(_e1, _e2, atol=1e-5) and not _m.training and not np.allclose(_e1, _p, atol=1e-6)
_t = copy.deepcopy(TEACHERS["shallow"]); _w0 = {n: p.detach().clone() for n, p in _t.named_parameters()}
recalibrate_bn(_t)
for n, p in _t.named_parameters():
    assert torch.equal(p, _w0[n])
print("probe and recalibration verified | evaluation rows:", len(_idx), "| raw structures:",
      all((IN33 / f"{a}_{m}_r40_removed_groups.csv").exists() for a in ARCHS for m in METHODS))


In [ ]:
# Stage 4 - the factorial: per-epoch dual-mode audit, final checkpoint saved, uniform recalibration
RUNS, EPOCHS, REG = OUT / "factorial_runs.csv", OUT / "factorial_epochs.csv", OUT / "checkpoint_registry.csv"
run_rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
ep_rows = pd.read_csv(EPOCHS).to_dict("records") if EPOCHS.exists() else []
reg_rows = pd.read_csv(REG).to_dict("records") if REG.exists() else []
done = {(r["architecture"], r["regime"], r["method"], r["seed"]) for r in run_rows}
print("complete cells:", len(done))

for arch, regime in RUN_CELLS:
    kind, n_ep = REGIMES[arch][regime]
    for method in METHODS:
        for seed in SEEDS:
            if (arch, regime, method, seed) in done:
                continue
            torch.manual_seed(seed); np.random.seed(seed)
            student = raw_student(arch, method); raw = audit(student, arch, "eval")
            loader = make_subset_loader(seed) if kind == "subset" else torch.utils.data.DataLoader(
                TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))
            opt = torch.optim.Adam(student.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W[arch])
            local_eps = []
            for ep in range(1, n_ep + 1):
                student.train()
                for xb, yb in loader:
                    opt.zero_grad(); lossf(student(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
                ev, bs = audit(student, arch, "eval"), audit(student, arch, "batch")
                col = is_collapse(ev, bs)
                local_eps.append({"architecture": arch, "regime": regime, "method": method, "seed": seed, "epoch": ep,
                                  **{f"eval_{k}": v for k, v in ev.items()}, **{f"batch_{k}": v for k, v in bs.items()},
                                  "is_collapse": col,
                                  "bn_var_max": float(max(m.running_var.max().item() for m in student.modules() if isinstance(m, nn.BatchNorm1d)))})
                if col:
                    torch.save({"state_dict": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()},
                                "architecture": arch, "method": method, "regime": regime, "seed": seed, "epoch": ep},
                               SNAP_DIR / f"{arch}_{method}_{regime}_s{seed}_e{ep}.pt")
            student.eval()
            sd = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            path = CKPT_DIR / f"{arch}_{method}_{regime}_s{seed}.pt"
            torch.save({"state_dict": sd, "architecture": arch, "method": method, "regime": regime, "seed": seed}, path)
            as_trained, batch_stats = local_eps[-1], None
            recalibrate_bn(student); recal = audit(student, arch, "eval")
            row = {"architecture": arch, "method": method, "regime": regime, "seed": seed}
            row.update({f"raw_{k}": v for k, v in raw.items()})
            row.update({f"trained_{k}": as_trained[f"eval_{k}"] for k in raw})
            row.update({f"batch_{k}": as_trained[f"batch_{k}"] for k in raw})
            row.update({f"recal_{k}": v for k, v in recal.items()})
            row["final_is_collapse"] = bool(as_trained["is_collapse"]); row["collapse_epochs"] = int(sum(e["is_collapse"] for e in local_eps))
            run_rows.append(row); ep_rows.extend(local_eps)
            reg_rows.append({"architecture": arch, "method": method, "regime": regime, "seed": seed,
                             "checkpoint": str(path.relative_to(REPO)), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
            pd.DataFrame(run_rows).to_csv(RUNS, index=False); pd.DataFrame(ep_rows).to_csv(EPOCHS, index=False)
            pd.DataFrame(reg_rows).to_csv(REG, index=False)
            print(f"{arch} {regime:11s} {method:9s} s{seed}: awbir trained={row['trained_awbir']:.4f} recal={row['recal_awbir']:.4f} | "
                  f"b2a trained={row['trained_b2a']:.4f} batch={row['batch_b2a']:.4f} recal={row['recal_b2a']:.4f}"
                  f"{'  COLLAPSE at ' + str([e['epoch'] for e in local_eps if e['is_collapse']]) if row['collapse_epochs'] else ''}")
runs = pd.DataFrame(run_rows); print("rows:", len(runs), "| cells with any collapse epoch:", int((runs.collapse_epochs > 0).sum()))


In [ ]:
# Stage 5 - unpruned control
UNP = OUT / "unpruned_control_epochs.csv"
rows = pd.read_csv(UNP).to_dict("records") if UNP.exists() else []
done = {(r["architecture"], r["seed"]) for r in rows}
for arch in ARCHS:
    for seed in SEEDS:
        if (arch, seed) in done:
            continue
        torch.manual_seed(seed); np.random.seed(seed)
        model = copy.deepcopy(TEACHERS[arch]).to(DEVICE); loader = make_subset_loader(seed)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W[arch]); hits = []
        for unit in range(1, E_MAX[arch] + 1):
            model.train()
            for xb, yb in loader:
                opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
            ev, bs = audit(model, arch, "eval"), audit(model, arch, "batch"); col = is_collapse(ev, bs)
            rows.append({"architecture": arch, "seed": seed, "unit": unit, "eval_b2a": ev["b2a"], "batch_b2a": bs["b2a"],
                         "eval_family_f1": ev["family_f1"], "batch_family_f1": bs["family_f1"], "eval_awbir": ev["awbir"], "is_collapse": col,
                         "bn_var_max": float(max(m.running_var.max().item() for m in model.modules() if isinstance(m, nn.BatchNorm1d)))})
            if col:
                hits.append(unit)
                torch.save({"state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "architecture": arch,
                            "method": "unpruned", "regime": "control", "seed": seed, "epoch": unit}, SNAP_DIR / f"{arch}_unpruned_control_s{seed}_e{unit}.pt")
        pd.DataFrame(rows).to_csv(UNP, index=False)
        print(f"unpruned {arch} s{seed}: normalisation collapse at {hits or 'none'}")
unp = pd.DataFrame(rows); print("unpruned collapse epochs:", int(unp.is_collapse.sum()))


In [ ]:
# Stage 6 - remedy on every collapse snapshot (factorial and control)
rows = []
for path in sorted(SNAP_DIR.glob("*.pt")):
    p = torch.load(path, map_location="cpu", weights_only=False)
    arch = p["architecture"]
    model = copy.deepcopy(TEACHERS[arch]) if p["method"] == "unpruned" else raw_student(arch, p["method"])
    model.load_state_dict(p["state_dict"]); model = model.to(DEVICE).eval()
    before, batch = audit(model, arch, "eval"), audit(model, arch, "batch")
    w0 = {n: t.detach().clone() for n, t in model.named_parameters()}
    recalibrate_bn(model)
    for n, t in model.named_parameters():
        assert torch.equal(t, w0[n]), f"recalibration altered weight {n}"
    after = audit(model, arch, "eval")
    rows.append({"snapshot": path.name, "architecture": arch, "method": p["method"], "regime": p["regime"], "seed": p["seed"], "epoch": p["epoch"],
                 **{f"before_{k}": v for k, v in before.items()}, **{f"batch_{k}": v for k, v in batch.items()},
                 **{f"after_{k}": v for k, v in after.items()},
                 "remedied": bool(after["b2a"] <= batch["b2a"] + 0.02)})
    print(f"{path.name}: b2a {before['b2a']:.3f} (batch {batch['b2a']:.3f}) -> {after['b2a']:.3f} | a2b {before['a2b']:.3f} -> {after['a2b']:.3f} | "
          f"famF1 {before['family_f1']:.3f} -> {after['family_f1']:.3f} | {'remedied' if rows[-1]['remedied'] else 'NOT remedied'}")
rem = pd.DataFrame(rows); rem.to_csv(OUT / "remedy.csv", index=False)
print("collapse snapshots:", len(rem), "| remedied:", int(rem.remedied.sum()) if len(rem) else 0)


In [ ]:
# Stage 7 - analysis under both normalisations
runs = pd.read_csv(OUT / "factorial_runs.csv"); rng = np.random.default_rng(0)


def shares(x):
    grand = x.mean(); ss_tot = ((x - grand) ** 2).sum()
    ss_m = x.shape[0] * ((x.mean(axis=0) - grand) ** 2).sum(); ss_s = x.shape[1] * ((x.mean(axis=1) - grand) ** 2).sum()
    return float(ss_m / ss_tot), float(ss_s / ss_tot), float((ss_tot - ss_m - ss_s) / ss_tot)


def perm_p(x, B=10000):
    m_obs, s_obs, _ = shares(x); s_null, m_null = [], []
    for _ in range(B):
        xs = x.copy()
        for j in range(xs.shape[1]): xs[:, j] = rng.permutation(xs[:, j])
        s_null.append(shares(xs)[1])
        xm = x.copy()
        for i in range(xm.shape[0]): xm[i, :] = rng.permutation(xm[i, :])
        m_null.append(shares(xm)[0])
    return float(np.mean(np.array(m_null) >= m_obs)), float(np.mean(np.array(s_null) >= s_obs))


def rank_rho(piv):
    ranks = piv.rank(axis=1)
    return float(np.nanmean([ranks.loc[a].corr(ranks.loc[b], method="spearman") for a, b in itertools.combinations(ranks.index, 2)]))


def paired(piv):
    out = []
    for a, b in itertools.combinations(METHODS, 2):
        d = (piv[a] - piv[b]).values; m = d.mean(); se = d.std(ddof=1) / np.sqrt(len(d))
        out.append({"contrast": f"{a}-{b}", "mean": float(m), "ci_lo": float(m - 2.776 * se), "ci_hi": float(m + 2.776 * se)})
    return out


analysis, contrast_rows = {}, []
for arch, regime in RUN_CELLS:
    sub = runs[(runs.architecture == arch) & (runs.regime == regime)]
    if len(sub) < len(METHODS) * len(SEEDS):
        print("incomplete, skipped:", arch, regime); continue
    raw_spread = float(sub.groupby("method")["raw_awbir"].first().agg(lambda v: v.max() - v.min())); cell = {}
    for norm in ["trained", "recal"]:
        piv = sub.pivot_table(index="seed", columns="method", values=f"{norm}_awbir")[METHODS]
        m_sh, s_sh, i_sh = shares(piv.values); p_m, p_s = perm_p(piv.values); pc = paired(piv)
        for c in pc: contrast_rows.append({"architecture": arch, "regime": regime, "normalisation": norm, **c})
        cell[norm] = {"method_share": m_sh, "seed_share": s_sh, "interaction_residual": i_sh, "p_method": p_m, "p_seed": p_s,
                      "rank_rho": rank_rho(piv), "rei_awbir": float(1 - (piv.max(axis=1) - piv.min(axis=1)).mean() / raw_spread) if raw_spread > 0 else float("nan"),
                      "max_abs_paired_mean": float(max(abs(c["mean"]) for c in pc)),
                      "mean_awbir_by_method": {m: float(v) for m, v in piv.mean().items()}}
        for metric in ["hsr_balanced", "b2a", "a2b", "family_f1"]:
            pm = sub.pivot_table(index="seed", columns="method", values=f"{norm}_{metric}")[METHODS]
            cell[norm][f"spread_{metric}"] = float((pm.max(axis=1) - pm.min(axis=1)).mean()); cell[norm][f"mean_{metric}"] = float(pm.values.mean())
    cell["raw_spread_awbir"] = raw_spread; cell["cells_with_collapse"] = int((sub.collapse_epochs > 0).sum())
    analysis[f"{arch}/{regime}"] = cell
    print(f"{arch}/{regime}: seed p {cell['trained']['p_seed']:.3f}->{cell['recal']['p_seed']:.3f} | method p {cell['trained']['p_method']:.3f}->{cell['recal']['p_method']:.3f} | "
          f"REI {cell['trained']['rei_awbir']:.3f}->{cell['recal']['rei_awbir']:.3f} | rank rho {cell['recal']['rank_rho']:.2f} | max paired {cell['recal']['max_abs_paired_mean']:.4f} | raw spread {raw_spread:.3f}")
pd.DataFrame(contrast_rows).to_csv(OUT / "paired_contrasts_both_normalisations.csv", index=False)
json.dump(analysis, open(OUT / "both_normalisations_analysis.json", "w"), indent=2)


In [ ]:
# Stage 8 - verdict
analysis = json.load(open(OUT / "both_normalisations_analysis.json"))
eps = pd.read_csv(OUT / "factorial_epochs.csv"); unp = pd.read_csv(OUT / "unpruned_control_epochs.csv")
rem = pd.read_csv(OUT / "remedy.csv") if (OUT / "remedy.csv").exists() and (OUT / "remedy.csv").stat().st_size > 5 else pd.DataFrame()
complete = len(analysis) == len(RUN_CELLS)
B1 = all(c["recal"]["rei_awbir"] >= 0.70 for c in analysis.values())
B2a = sum(1 for c in analysis.values() if c["recal"]["p_seed"] < 0.05) >= 2
B2b = all(c["recal"]["max_abs_paired_mean"] < 0.03 for c in analysis.values())
B3 = all(c["recal"]["rank_rho"] < 0.5 for c in analysis.values())
n_col = int(eps.is_collapse.sum()) + int(unp.is_collapse.sum()); B4 = n_col > 0
B5 = bool(len(rem) and rem.remedied.all()) if B4 else None
verdict = {"arm": "B_second_corpus_factorial", "verdict_complete": bool(complete), "cells_analysed": list(analysis),
           "B1_equalisation": bool(B1), "B2a_seed_effect_detectable": bool(B2a), "B2b_selection_effects_small": bool(B2b),
           "B3_rank_instability": bool(B3), "B4_collapse_occurs": bool(B4), "B5_remedy": B5,
           "collapse_epochs_factorial": int(eps.is_collapse.sum()), "collapse_epochs_unpruned": int(unp.is_collapse.sum()),
           "collapse_cells": eps[eps.is_collapse][["architecture", "regime", "method", "seed", "epoch", "eval_b2a", "batch_b2a"]].round(4).to_dict("records"),
           "unpruned_collapse_cells": unp[unp.is_collapse][["architecture", "seed", "unit", "eval_b2a", "batch_b2a"]].round(4).to_dict("records"),
           "cells": {k: {n: {kk: vv for kk, vv in v[n].items() if kk != "mean_awbir_by_method"} for n in ("trained", "recal")}
                     | {"raw_spread_awbir": v["raw_spread_awbir"]} for k, v in analysis.items()},
           "remedy_summary": ({c: float(rem[c].median()) for c in ["before_b2a", "after_b2a", "before_a2b", "after_a2b", "before_hsr_balanced", "after_hsr_balanced"]} if len(rem) else None),
           "prereg": json.load(open(OUT / "B34_PREREGISTRATION.json"))}
(OUT / "B34_verdict.json").write_text(json.dumps(verdict, indent=2, default=lambda o: int(o) if isinstance(o, (np.integer,)) else float(o)))
print(json.dumps({k: v for k, v in verdict.items() if k not in ("prereg", "cells", "collapse_cells", "unpruned_collapse_cells")}, indent=2))
for k, v in verdict["cells"].items():
    print(f"{k:22s} REI {v['recal']['rei_awbir']:.3f} | seed p {v['recal']['p_seed']:.3f} | method p {v['recal']['p_method']:.3f} | rank rho {v['recal']['rank_rho']:.2f} | max paired {v['recal']['max_abs_paired_mean']:.4f}")


In [ ]:
# Stage 9 - figures
runs = pd.read_csv(OUT / "factorial_runs.csv"); analysis = json.load(open(OUT / "both_normalisations_analysis.json"))
cells_present = [c for c in RUN_CELLS if f"{c[0]}/{c[1]}" in analysis]
fig, axes = plt.subplots(1, len(cells_present), figsize=(4.6 * len(cells_present), 3.4), squeeze=False)
for ax, (arch, regime) in zip(axes[0], cells_present):
    sub = runs[(runs.architecture == arch) & (runs.regime == regime)]
    for method in METHODS:
        d = sub[sub.method == method].sort_values("seed")
        ax.plot(d.seed.astype(str), d.recal_awbir, marker="o", lw=1.1, label=method)
    ax.set_title(f"IoMT {arch} / {regime} (recalibrated)"); ax.set_xlabel("recovery seed"); ax.set_ylabel("AWBIR")
axes[0][0].legend(fontsize=6)
fig.tight_layout(); fig.savefig(OUT / "B34_awbir_by_seed.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7.2, 3.4)); labels = list(analysis); xs = np.arange(len(labels))
ax.bar(xs - 0.2, [analysis[k]["trained"]["seed_share"] for k in labels], 0.4, label="seed share, as-trained", color="#dd8452")
ax.bar(xs + 0.2, [analysis[k]["recal"]["seed_share"] for k in labels], 0.4, label="seed share, recalibrated", color="#c44e52")
ax.scatter(xs - 0.2, [analysis[k]["trained"]["method_share"] for k in labels], marker="s", color="k", label="method share, as-trained")
ax.scatter(xs + 0.2, [analysis[k]["recal"]["method_share"] for k in labels], marker="D", color="0.4", label="method share, recalibrated")
ax.set_xticks(xs); ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=8); ax.set_ylabel("share of AWBIR variance"); ax.legend(fontsize=7)
ax.set_title("IoMT: variance shares before and after uniform recalibration"); fig.tight_layout(); fig.savefig(OUT / "B34_variance_shares.png", dpi=200); plt.show()

eps = pd.read_csv(OUT / "factorial_epochs.csv"); unp = pd.read_csv(OUT / "unpruned_control_epochs.csv")
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.2))
for ax, arch in zip(axes, ["shallow", "deep"]):
    for seed in SEEDS:
        d = unp[(unp.architecture == arch) & (unp.seed == seed)].sort_values("unit")
        ax.plot(d.unit, d.eval_b2a - d.batch_b2a, marker="o", lw=1.1, label=f"seed {seed}")
    ax.axhline(COLLAPSE_GAP, color="0.4", ls=":", lw=0.9); ax.set_title(f"IoMT unpruned {arch}: eval minus batch benign escalation")
    ax.set_xlabel("unit"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "B34_unpruned_control.png", dpi=200); plt.show()
print("figures written ->", OUT)


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)